In [ ]:
USE {
    repositories {
        mavenCentral()
        maven("https://central.sonatype.com/repository/maven-snapshots/")
    }
    dependencies {
        implementation("io.github.kotlinrl:integration:0.1.0-SNAPSHOT")
    }
}

In [ ]:
import io.github.kotlinrl.core.*
import io.github.kotlinrl.integration.gymnasium.*
import io.github.kotlinrl.integration.gymnasium.GymnasiumEnvs.*
import io.github.kotlinrl.core.wrapper.*
import java.io.*

In [ ]:
val folder = "videos/blackjack"
val f = File(folder)
deleteRecursively(f)
f.mkdirs()

val env = gymnasium.make<BlackjackEnv>(Blackjack_v1, seed=123, render=true)

var wins = 0.0
var losses = 0.0
var draws = 0.0
var fameIndex = 0
for(i in 0 until 100) {
    var episodeOver = false
    var episodeReward = 0.0
    var (state, _) = env.reset()
    saveFrameAsPng(env.render() as RenderFrame, folder, 1, fameIndex++)
    println("Starting state: $state")
    while (!episodeOver) {
        val (playerSum: Int, dealerSum: Int, _) = state.map { it as Int }
        val action = if (playerSum < dealerSum) 1 else 0
        println("Action: ${if (action == 0) "Stick" else "Hit"}")
        val (nextState, reward, terminated, truncated, _) = env.step(action)
        saveFrameAsPng(env.render() as RenderFrame, folder, 1, fameIndex++)
        println("State: $nextState, Reward: $reward")
        episodeReward += reward
        episodeOver = terminated || truncated
        state = nextState
    }
    if (episodeReward > 0) wins++ else if (episodeReward < 0) losses++ else draws++
    println("Episode finished!: Episode reward: $episodeReward")
}
println("Wins: $wins, Losses: $losses, Draws: $draws")
env.close()
saveEpisodeAsMp4JCodec(folder, 1)
displayVideo(
    episode = 1,
    folder = folder
)